### Data Ingestion

### Data Ingestion Pipeline Overview

This notebook implements a complete data ingestion pipeline for RAG (Retrieval-Augmented Generation) systems:

1. **PDF Text Extraction**: Load PDF documents and extract text content page by page
2. **Text Preprocessing**: Clean and normalize extracted text
3. **Statistical Analysis**: Calculate character, word, and token counts for each page
4. **Sentence Segmentation**: Use spaCy to split text into individual sentences
5. **Text Chunking**: Group sentences into meaningful chunks for embedding
6. **Chunk Statistics**: Analyze chunk sizes and quality metrics
7. **Data Quality Inspection**: Review short chunks and sample data

Each step includes inline comments explaining the purpose and implementation details.

In [57]:
!pip install ragas

In [58]:
# Cell: PDF Text Extraction and Dataset Preparation
# Purpose: Extract text from PDF documents page-by-page with statistics and prepare for NLP processing
# Key functions:
#   - text_formatter: Cleans extracted text by removing newlines and normalizing whitespace
#   - open_and_read_pdf(pdf_path): Main extraction function that processes entire PDF
# Output: List of dicts containing page_number (1-based), text content, and statistics
# Statistics include: page_char_count, page_word_count, page_sentence_count_raw, page_token_count

import fitz  # PyMuPDF library for PDF text extraction (better than PyPDF for text handling)
from tqdm.auto import tqdm  # Progress bars for long-running operations

# Text preprocessing function to clean and normalize extracted text
def text_formatter(text: str) -> str:
    """Remove newlines and normalize whitespace in extracted PDF text."""
    text = text.replace("\n", " ")  # Replace newlines with spaces
    text = " ".join(text.split())  # Normalize multiple spaces to single space
    return text

# Main PDF extraction function
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Extract text and metadata from each page of a PDF file.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        List of dicts with page_number, text, and statistics (char/word/token counts)
    """
    pages_and_texts = []
    
    doc = fitz.open(pdf_path)  # Open PDF using PyMuPDF
    
    for page_num, page in enumerate(tqdm(doc)):
        text = page.get_text()  # Extract text from page
        text = text_formatter(text)  # Clean text
        
        pages_and_texts.append({
            "page_number": page_num + 1,  # 1-based page numbering (human-readable)
            "page_char_count": len(text),
            "page_word_count": len(text.split()),
            "page_sentence_count_raw": len(text.split(".")),  # Rough estimate
            "page_token_count": len(text) // 4,  # Approximate: 1 token ≈ 4 chars
            "text": text
        })
    
    doc.close()
    return pages_and_texts

# Extract text from the PDF
pdf_path = "../data/human-nutrition-text.pdf"
pages_and_texts = open_and_read_pdf(pdf_path)

# Display extraction summary
print(f"Extracted {len(pages_and_texts)} pages from PDF")
print(f"Average page length: {sum(p['page_word_count'] for p in pages_and_texts) / len(pages_and_texts):.0f} words")

In [59]:
# Random sampling to inspect data quality and structure
import random 

random.sample(pages_and_texts,k=2)

In [60]:
# Convert page data to pandas DataFrame for easier analysis
import pandas as pd
df=pd.DataFrame(pages_and_texts)
df.head()


In [61]:
# Generate statistical summary of page data
df.describe().round(2)

In [62]:
# Cell: Initialize spaCy NLP Pipeline
# Purpose: Set up sentence segmentation for breaking text into individual sentences
# Key components:
#   - nlp: spaCy blank English language model (no pretrained weights)
#   - sentencizer: Rule-based component that splits text at sentence boundaries (. ! ?)
# Output: nlp object ready for sentence segmentation on all pages
# Why spaCy? Much better than simple .split(".") - handles abbreviations, decimals, etc.

from spacy.lang.en import English  # Import English language model

nlp = English()  # Create blank English language model instance (lightweight, no word vectors needed)

# Add sentencizer pipeline component for rule-based sentence boundary detection
nlp.add_pipe("sentencizer")

In [63]:
# Cell: Apply Sentence Segmentation to All Pages
# Purpose: Split each page's text into individual sentences using spaCy's rule-based sentencizer
# Process:
#   1. For each page in pages_and_texts
#   2. Pass text through nlp pipeline to get sentence objects (Doc.sents)
#   3. Convert sentence objects to strings for simpler downstream processing
#   4. Count sentences per page for quality analysis
# Output: pages_and_texts updated with "sentences" list and "page_sentence_count_spacy"

from tqdm import tqdm  # Import tqdm for progress tracking

# Process each page to split text into individual sentences using spaCy
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)  # Segment page text into sentence objects
    
    # Convert spaCy sentence objects to strings for easier downstream processing
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    
    # Count sentences per page using spaCy's rule-based segmentation (more accurate than raw split)
    item["page_sentence_count_spacy"] = len(item["sentences"])

In [64]:
#inspect an example page
random.sample(pages_and_texts, k=1)

In [65]:
df=pd.DataFrame(pages_and_texts)
df.describe().round(2)

In [66]:
# Cell: Text Chunking - Group Sentences into Retrievable Chunks
# Purpose: Group individual sentences into fixed-size chunks for efficient embedding and retrieval
# Key parameters:
#   - num_sentence_chunk_size = 10: Group 10 consecutive sentences into one "chunk"
#   - Chunks smaller than 30 tokens will be filtered out later
# Process:
#   1. For each page, use split_list() helper to group sentences into chunks
#   2. Store resulting chunk list and count in page record
# Output: pages_and_texts updated with "sentence_chunks" (list of lists) and "num_chunks"
# Why fixed-size? Makes retrieval consistent - not too long (confusing), not too short (no context)

# Text chunking configuration
num_sentence_chunk_size = 10  # Adjust this to control chunk granularity

# Utility function to split lists into fixed-size sublists
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Splits input_list into sublists of specified slice_size.
    
    Example: split_list([1,2,3,4,5,6,7], 3) → [[1,2,3], [4,5,6], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Apply sentence chunking to each page

from tqdm import tqdm

for item in tqdm(pages_and_texts):
    # Split page sentences into chunks of specified size
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    
    # Count how many chunks were created for this page
    item["num_chunks"] = len(item["sentence_chunks"])

In [67]:
random.sample(pages_and_texts, k=1)

In [72]:
# Cell: Flatten Chunks into Individual Records
# Purpose: Convert hierarchical page->sentences->chunks structure into flat list of retrievable chunk records
# Each chunk becomes one record with: page_number, sentence_chunk (text), chunk_char_count, chunk_word_count, chunk_token_count
# Key calculations:
#   - chunk_char_count: Length of chunk text (used for filtering and analysis)
#   - chunk_word_count: Word count using basic split() (for metadata)
#   - chunk_token_count: Approximate token count (length/4, typical token ≈ 4 characters)
# Output: pages_and_chunks list contains all chunks ready for embedding and retrieval
# Important: This is the first time we flatten - later filtering will create pages_and_chunks_over_min_token_len

# Convert sentence chunks into individual records for embedding and retrieval
import re  # Import regex for text cleaning

# Initialize list to store final processed chunks
pages_and_chunks = []

# Convert each sentence chunk to a flattened record with statistics
from tqdm import tqdm

for item in tqdm(pages_and_texts):
    for chunk_index, sentence_chunk in enumerate(item["sentence_chunks"]):
        # Join sentences in chunk with space
        joined_sentence_chunk = " ".join(sentence_chunk)
        
        # Calculate chunk statistics
        chunk_char_count = len(joined_sentence_chunk)
        chunk_word_count = len(joined_sentence_chunk.split())
        chunk_token_count = chunk_char_count // 4  # Approximate: 1 token ≈ 4 chars
        
        pages_and_chunks.append({
            "page_number": item["page_number"],
            "sentence_chunk": joined_sentence_chunk,
            "chunk_char_count": chunk_char_count,
            "chunk_word_count": chunk_word_count,
            "chunk_token_count": chunk_token_count,
        })

print(f"Created {len(pages_and_chunks)} total chunks from {len(pages_and_texts)} pages")

In [73]:
random.sample(pages_and_chunks, k=1)

In [74]:
# Analyze chunk statistics and distribution
df = pd.DataFrame(pages_and_chunks)  # Convert chunks to DataFrame for analysis
df.describe().round(2)  # Generate statistical summary of chunk metrics

In [75]:
# Cell: Data Quality Inspection - Identify Short/Problematic Chunks
# Purpose: Examine chunks that are too short to carry meaningful semantic information
# Quality threshold:
#   - min_token_length = 30: Chunks with < 30 tokens (~120 chars) are too short for good embeddings
#   - Rationale: Very short chunks don't have enough context for semantic similarity search
# Action: Sample and display 5 short chunks to understand data quality issues
# Output: Printed examples of short chunks for manual inspection

# Data quality inspection: examine chunks that are too short for effective retrieval
import pandas as pd

# Convert chunks to DataFrame for easier filtering
df = pd.DataFrame(pages_and_chunks)

min_token_length = 30  # Minimum token threshold for meaningful chunks

# Sample and display 5 random chunks that are shorter than the minimum threshold
print(f"Chunks below {min_token_length} token threshold:")
print("=" * 80)
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Tokens: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"][:100]}...')
    print("-" * 80)

In [76]:
# Cell: Filter Chunks by Quality Threshold
# Purpose: Remove short/low-quality chunks (< 30 tokens) to ensure high-quality embeddings
# Process:
#   1. Filter chunks where chunk_token_count > min_token_length (30)
#   2. Convert filtered dataframe to list of dicts (orient="records")
# Output: pages_and_chunks_over_min_token_len - final list of quality chunks for embedding and retrieval
# IMPORTANT: This filtered list is used for ALL downstream embedding and retrieval operations
# Not the original pages_and_chunks!

pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")

print(f"Filtered {len(pages_and_chunks)} → {len(pages_and_chunks_over_min_token_len)} chunks")
print(f"Removed {len(pages_and_chunks) - len(pages_and_chunks_over_min_token_len)} low-quality chunks ({100 * (len(pages_and_chunks) - len(pages_and_chunks_over_min_token_len)) / len(pages_and_chunks):.1f}%)")
print("\nFirst 2 quality chunks:")
for i, chunk in enumerate(pages_and_chunks_over_min_token_len[:2], 1):
    print(f"  [{i}] Tokens: {chunk['chunk_token_count']:3d} | Page: {chunk['page_number']:3d} | Text: {chunk['sentence_chunk'][:50]}...")

In [77]:
# Cell: Initialize Embedding Model and Demo (Educational Reference)
# Purpose: Load SentenceTransformer model for converting text to semantic embeddings + demonstrate on sample sentences
# Model: all-mpnet-base-v2
#   - 768-dimensional embeddings (semantic vectors)
#   - Trained on semantic similarity tasks
#   - Works well for retrieval and clustering
# Device: CPU (efficient for inference, no GPU required)
# Note: This cell demonstrates embedding on demo sentences ONLY
#       Actual chunk embedding happens in the next cells using pages_and_chunks_over_min_token_len
#
# Parameters:
#   - model_name_or_path: 'all-mpnet-base-v2' - Multilingual pretrained BERT for embeddings
#   - device: 'cpu' - Use CPU for inference (portable, no GPU hardware required)
# Output: embedding_model object ready for encoding queries (reused later)

from sentence_transformers import SentenceTransformer

# Load embedding model on CPU
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device="cpu")  # CPU-based inference

print("Embedding model loaded successfully!")
print(f"Model: {embedding_model.get_sentence_embedding_dimension()}-dimensional embeddings")

# Demo: Create a list of example sentences to demonstrate embedding API
sentences = [
    "The Sentence Transformers library provides an easy and open-source way to create embeddings.",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning!",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer."
]

# Sentences are encoded/embedded by calling model.encode()
# Returns numpy arrays of shape (num_sentences, 768)
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

# Display embedding results
print("\n" + "="*80)
print("DEMO: Embedding 4 sample sentences")
print("="*80)
for sentence, embedding in embeddings_dict.items():
    print(f"Sentence: {sentence}")
    print(f"Embedding shape: {embedding.shape}")
    print(f"First 5 dims: {embedding[:5]}")
    print()

In [80]:
# Cell: Generate Embeddings for All Quality Chunks (CPU-based, Sequential)
# Purpose: Convert each chunk text into 768-dimensional semantic vector using SentenceTransformer
# Process:
#   1. Load model with device='cpu' if not already loaded
#   2. Iterate through each quality chunk (pages_and_chunks_over_min_token_len)
#   3. Encode chunk text to 768-dim vector and store in 'embedding' field
# Parameters:
#   - Embedding method: One chunk at a time (simple, memory efficient but slower)
#   - Device: CPU (no GPU required)
# Output: Each chunk dict gets an 'embedding' numpy array field (768 dims)
# Note: If only this cell runs (skipping batch cell), embeddings still available but slower

# CPU-based embeddings - Sequential processing (one chunk at a time)
import time

# Ensure the embedding model is defined in this cell in case previous cells were not executed
try:
    embedding_model
except NameError:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                          device="cpu")

from tqdm import tqdm

# Ensure model is on CPU before encoding
embedding_model.to("cpu")

print(f"Encoding {len(pages_and_chunks_over_min_token_len)} chunks sequentially...")
start_time = time.time()

# Embed each chunk one by one and store embeddings in the chunk record
for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

elapsed_time = time.time() - start_time
print(f"\nSequential encoding completed in {elapsed_time:.2f} seconds")

In [ ]:
# Cell: Save Processed Chunks to CSV
# Purpose: Persist all quality chunks with metadata to disk for later analysis/reloading
# Note: Embeddings are NOT saved to CSV (too large, lose precision). Keep embeddings in memory only.
# Output file: text_chunks_and_embeddings_df.csv
# Fields saved: page_number, sentence_chunk, chunk_char_count, chunk_word_count, chunk_token_count

import pandas as pd

# Create DataFrame from chunk records
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)

# Drop embedding column before saving (too large for CSV, numpy arrays can't be CSV-ified well)
if 'embedding' in text_chunks_and_embeddings_df.columns:
    text_chunks_and_embeddings_df = text_chunks_and_embeddings_df.drop('embedding', axis=1)

embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

print(f"✓ Saved {len(text_chunks_and_embeddings_df)} quality chunks to {embeddings_df_save_path}")
print(f"  File size: ~{text_chunks_and_embeddings_df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

In [ ]:
# Cell: Verify Saved Chunks
# Purpose: Confirm successful save to disk and display sample of persisted chunk data
# Note: Embeddings are NOT in this file (kept in memory only in text_chunk_embeddings tensor)

import pandas as pd

# Import saved file and display stats
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
print(f"✓ Loaded {len(text_chunks_and_embedding_df_load)} chunks from {embeddings_df_save_path}")
print(f"\nColumn names: {list(text_chunks_and_embedding_df_load.columns)}")
print(f"\nFirst 3 chunks:")
print(text_chunks_and_embedding_df_load.head(3))

In [ ]:
# Cell: Ensure Embedding Model Ready for Query
# Purpose: Load embedding model if not already in memory (supports running cells independently)
# Device: CPU (consistent with all embedding operations)
# Action: This cell ensures model is available for semantic search operations that follow

from sentence_transformers import SentenceTransformer, util

try:
    embedding_model  # Check if already loaded
except NameError:
    print("Loading embedding model...")
    embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                          device="cpu")  # CPU-based inference
    print("✓ Model loaded and ready for query encoding")

In [ ]:
# Cell: Retrieve Relevant Chunks by Semantic Similarity
# Purpose: Find top-k document chunks most similar to user query using semantic search
# Process:
#   1. Define query string (natural language question/intent)
#   2. Embed query using same SentenceTransformer model used for chunks (all-mpnet-base-v2)
#   3. Compute cosine similarity between query and all chunk embeddings (dot product on L2-normalized vectors)
#   4. Return top-k most relevant chunks ranked by similarity score (0.0-1.0 range)
# Parameters:
#   - query: User's semantic search question (any natural language text)
#   - k=5: Number of top results to return
#   - text_chunk_embeddings: Tensor of all chunk embeddings from earlier batch cell
# Output: top_results_dot_product contains (scores, indices) of top-5 chunks
# Note: This uses the REAL chunk embeddings from pages_and_chunks_over_min_token_len (not demo data!)

# Define a semantic search query
query = "macronutrients functions"
print(f"Query: {query}\n")

# Step 1: Embed the query to the same numerical space as the document chunks
# Note: Must use same embedding model (all-mpnet-base-v2) and device (CPU) for compatibility
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# Step 2: Compute similarity scores using dot product on normalized embeddings
# (embeddings are L2-normalized, so dot product = cosine similarity, range [0, 1])
from time import perf_counter as timer
import torch

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=text_chunk_embeddings)[0]
end_time = timer()

print(f"Similarity search time on {len(text_chunk_embeddings)} chunks: {end_time - start_time:.5f} seconds.")

# Step 3: Get top-k results (cap at 5, but never exceed available chunks)
k = min(5, dot_scores.numel())  # Dynamic k prevents out-of-range errors
if k > 0:
    top_results_dot_product = torch.topk(dot_scores, k=k)
    print(f"Found {k} relevant chunks (max score: {top_results_dot_product[0][0]:.4f})")
else:
    print("ERROR: No chunks available for retrieval")

In [ ]:
# Cell: Current Embedding Statistics and Scaling Info
# Purpose: Display embedding statistics and performance benchmarks for current dataset
# Note: Demonstrates current system can handle real document embeddings (not just demo data)
# Scaling notes: For larger datasets, consider GPU acceleration or hierarchical retrieval

print(f"\n{'='*80}")
print("Embedding Statistics & Scaling Info")
print(f"{'='*80}\n")

print(f"Total chunks with embeddings: {len(text_chunk_embeddings)}")
print(f"Embedding dimensions: {text_chunk_embeddings.shape[1]}")
print(f"Total embedding memory: ~{text_chunk_embeddings.shape[0] * text_chunk_embeddings.shape[1] * 4 / 1024 / 1024:.1f} MB")
print(f"\n✓ This CPU-based setup supports semantic search across all chunks")
print(f"\nPerformance notes:")
print(f"  - Current query: {end_time - start_time:.5f} seconds on {len(text_chunk_embeddings)} chunks")
print(f"  - For 10k chunks: ~{(end_time - start_time) * 10:.2f}s")
print(f"  - For 100k+ chunks: Consider GPU acceleration or multi-stage retrieval")

In [ ]:
# Cell: Display Retrieved Results
# Purpose: Show top-k most relevant chunks ranked by semantic similarity to query
# Output for each result:
#   - Rank and Similarity score (0.0-1.0, higher = more relevant)
#   - Full chunk text (wrapped to 80 chars per line for readability)
#   - Source page number in original PDF
# Note: Results are from REAL chunk embeddings (text_chunk_embeddings), not demo data

if 'top_results_dot_product' in locals():
    print(f"\n{'='*80}")
    print(f"Query Results (Top {len(top_results_dot_product[1])} Chunks)")
    print(f"{'='*80}")
    print(f"Query: '{query}'\n")
    
    # Loop through scores and indices from torch.topk
    for rank, (score, idx) in enumerate(zip(top_results_dot_product[0], top_results_dot_product[1]), 1):
        print(f"[Rank {rank}] Similarity Score: {score:.4f}")
        print("-" * 80)
        
        print("Text:")
        import textwrap
        wrapped_text = textwrap.fill(pages_and_chunks_over_min_token_len[idx]["sentence_chunk"], width=80)
        print(wrapped_text)
        
        print(f"\nSource Page: {pages_and_chunks_over_min_token_len[idx]['page_number']}")
        print("="*80 + "\n")
else:
    print("No results available - run semantic search cell first")

In [ ]:
# Cell: Display Source PDF Page
# Purpose: Render the actual PDF page containing the most relevant chunk
# Process:
#   1. Get page_number from top-ranked chunk metadata
#   2. Load original PDF and render page to high-resolution image (300 DPI)
#   3. Display image in notebook with title showing query and page number
# Parameters:
#   - pdf_path: Path to source PDF file
#   - dpi=300: Resolution for rendering (300 DPI = high quality, readable text)
#   - figsize=(13, 10): Image display size in inches (notebook display)
# Note: Requires PDF file at ../data/human-nutrition-text.pdf
# Importance: Closes the loop - verifies retrieved chunks are actually relevant by showing context

if 'top_results_dot_product' in locals():
    import fitz  # PyMuPDF
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Path to source PDF
    pdf_path = "../data/human-nutrition-text.pdf"
    
    try:
        doc = fitz.open(pdf_path)
        
        # Get page number from top retrieval result (use REAL page_number from metadata)
        try:
            selected_page_number = pages_and_chunks_over_min_token_len[top_results_dot_product[1][0]]["page_number"]
        except Exception as e:
            print(f"Error retrieving page number: {e}, defaulting to page 1")
            selected_page_number = 1
        
        # Convert to zero-based index for PDF API (PDF page 1 = index 0)
        pdf_page_index = max(0, selected_page_number - 1)
        page = doc.load_page(pdf_page_index)
        
        # Render page to high-resolution image (300 DPI for readability)
        img = page.get_pixmap(dpi=300)
        
        doc.close()
        
        # Convert Pixmap to numpy array for matplotlib display
        img_array = np.frombuffer(img.samples_mv, dtype=np.uint8).reshape((img.h, img.w, img.n))
        
        # Display image with context
        plt.figure(figsize=(13, 10))
        plt.imshow(img_array)
        plt.title(f"Top Retrieval Result | Query: '{query}' | Page {selected_page_number}", fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
    except FileNotFoundError:
        print(f"ERROR: PDF file not found at {pdf_path}")
        print("Ensure the nutrition textbook is downloaded to ../data/")
else:
    print("No results available - run semantic search cell first")

In [ ]:
# Cell: Display Source PDF Page
# Purpose: Render the actual PDF page containing the most relevant chunk
# Process:
#   1. Get page_number from top-ranked chunk metadata
#   2. Load PDF and render page to high-resolution image (300 DPI)
#   3. Display image in notebook with title showing query and page number
# Parameters:
#   - pdf_path: Path to source PDF file
#   - dpi=300: Resolution for rendering (high quality)
#   - figsize=(13, 10): Image display size in inches
# Note: Requires PDF file at ../data/human-nutrition-text.pdf

if 'top_results_dot_product' in locals():
    import fitz  # PyMuPDF
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Open PDF and select the page using real page_number metadata
    pdf_path = "../data/human-nutrition-text.pdf"  # requires PDF to be downloaded
    
    try:
        doc = fitz.open(pdf_path)
        
        # Get page number from top retrieval result
        try:
            selected_page_number = pages_and_chunks_over_min_token_len[top_results_dot_product[1][0]]["page_number"]
        except Exception:
            selected_page_number = 1
        
        # Convert to zero-based index for PDF API
        pdf_page_index = max(0, selected_page_number - 1)

        page = doc.load_page(pdf_page_index)    print("No results available - run semantic search cell first")

        else:

        # Render page to high-resolution image        print(f"PDF file not found at {pdf_path}")

        img = page.get_pixmap(dpi=300)  # 300 DPI for readability    else:

                plt.show()

        doc.close()        plt.tight_layout()

                plt.axis('off')

        # Convert Pixmap to numpy array for display        plt.title(f"Query: '{query}' | Page {selected_page_number}", fontsize=14)

        img_array = np.frombuffer(img.samples_mv, dtype=np.uint8).reshape((img.h, img.w, img.n))        plt.imshow(img_array)

                plt.figure(figsize=(13, 10))
        # Display image with context

In [ ]:
# Cell: Utility Functions - Similarity Metrics (Reference)
# Purpose: Demonstrate dot product and cosine similarity calculations for educational reference
# Note: In practice, sentence_transformers.util.dot_score() is used (optimized C++/CUDA backend)
#
# Similarity Metrics:
#   - Dot product: a·b = sum(a[i] * b[i]) 
#     * Fast if vectors already L2-normalized
#     * With all-mpnet embeddings: dot_product ≈ cosine_similarity
#   - Cosine similarity: (a·b) / (||a|| * ||b||) 
#     * Always normalized to [-1, 1]
#     * Slower but works with non-normalized vectors
# Use Cases:
#   - Dot product: Fast similarity when embeddings are L2-normalized (our case)
#   - Cosine similarity: When normalizing non-normalized vectors
#
# For this pipeline: Use dot_score() (optimized), not these functions (for education only)

import torch

def dot_product(vector1, vector2):
    """Compute dot product of two vectors (fastest for normalized embeddings)."""
    return torch.dot(vector1, vector2)

def cosine_similarity(vector1, vector2):
    """
    Compute cosine similarity between two vectors (normalized to [-1, 1]).
    Works with both normalized and non-normalized vectors.
    """
    dot_product_val = torch.dot(vector1, vector2)
    
    # L2 norms (magnitude of each vector)
    norm_vector1 = torch.sqrt(torch.sum(vector1 ** 2))
    norm_vector2 = torch.sqrt(torch.sum(vector2 ** 2))
    
    # Avoid division by zero
    if norm_vector1 == 0 or norm_vector2 == 0:
        return 0.0
    
    return dot_product_val / (norm_vector1 * norm_vector2)

# Educational Example: Demonstrate similarity metrics with sample vectors
print("\n" + "="*80)
print("Similarity Metrics Examples")
print("="*80)

vector1 = torch.tensor([1, 2, 3], dtype=torch.float32)  # Reference vector
vector2 = torch.tensor([1, 2, 3], dtype=torch.float32)  # Identical
vector3 = torch.tensor([4, 5, 6], dtype=torch.float32)  # Correlated
vector4 = torch.tensor([-1, -2, -3], dtype=torch.float32)  # Opposite direction

print("\nv1 = [1, 2, 3]  (reference)")
print(f"  ·  v2 [1, 2, 3] (identical):  dot={dot_product(vector1, vector2):.2f}, cos={cosine_similarity(vector1, vector2):.4f}")
print(f"  ·  v3 [4, 5, 6] (correlated): dot={dot_product(vector1, vector3):.2f}, cos={cosine_similarity(vector1, vector3):.4f}")
print(f"  ·  v4 [-1,-2,-3] (opposite):  dot={dot_product(vector1, vector4):.2f}, cos={cosine_similarity(vector1, vector4):.4f}")
print("\nNote: all-mpnet embeddings are L2-normalized, so dot_product ≈ cosine_similarity")

In [ ]:
# Cell: Core Retrieval Function - Semantic Search
# Purpose: Reusable function to retrieve top-k chunks most similar to any query
# This is the heart of the RAG system - can be called from agents, APIs, etc.
#
# Function: retrieve_relevant_resources()
# 
# Parameters:
#   - query (str): User's search query (any natural language text, e.g., "symptoms of pellagra")
#   - embeddings (torch.tensor): Precomputed chunk embeddings matrix of shape (N x 768)
#   - model (SentenceTransformer): Model for encoding query (must match chunk embedding model)
#   - n_resources_to_return (int): Number of top results to return (default 5)
#   - print_time (bool): Whether to print elapsed time statistics (default True)
#
# Returns:
#   - scores: Similarity scores for top-k chunks (torch.tensor)
#   - indices: Indices of top-k chunks in embeddings array (torch.tensor)
#
# Process:
#   1. Encode query using same model as chunks (all-mpnet-base-v2)
#   2. Compute cosine similarity via dot product on normalized vectors
#   3. Get top-k results using torch.topk (dynamic k prevents out-of-range errors)
#   4. Return (scores, indices) for downstream ranking/display
#
# CPU-friendly: Fast even without GPU, scales to ~100k chunks

from sentence_transformers import SentenceTransformer, util
from time import perf_counter as timer
import torch

def retrieve_relevant_resources(
    query: str,
    embeddings: torch.tensor,
    model: SentenceTransformer,
    n_resources_to_return: int = 5,
    print_time: bool = True
):
    """
    Retrieves top-k most similar chunks to a query using semantic similarity.
    
    Args:
        query (str): Natural language search query
        embeddings (torch.tensor): Tensor of precomputed chunk embeddings (num_chunks x 768)
        model (SentenceTransformer): SentenceTransformer model for encoding query
        n_resources_to_return (int): Number of top chunks to return (default 5)
        print_time (bool): Whether to print search time statistics (default True)
    
    Returns:
        scores (torch.tensor): Top-k similarity scores (0.0-1.0 range)
        indices (torch.tensor): Top-k chunk indices in embeddings array
    
    Raises:
        No exceptions - returns empty tensors if no embeddings available
    """

    # Step 1: Embed the query to the same vector space as chunks
    # Uses same model and device for consistency
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Step 2: Compute similarity scores (dot product on normalized embeddings = cosine similarity)
    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Similarity computation: {end_time - start_time:.5f}s on {len(embeddings)} embeddings")

    # Step 3: Top-k retrieval with safety check
    # Never request more than available embeddings (prevents torch.topk out-of-range error)
    k = min(n_resources_to_return, dot_scores.numel())
    if k == 0:
        if print_time:
            print("[WARNING] No embeddings available for retrieval")
        return torch.tensor([]), torch.tensor([])

    scores, indices = torch.topk(input=dot_scores, k=k)

    return scores, indices

In [ ]:
# Cell: Result Formatting and Display Functions
# Purpose: Pretty-print and format semantic search results for easy human reading
# Functions provided:
#   1. print_wrapped(text, wrap_length=80): Format long text across multiple lines
#   2. print_top_results_and_scores(...): Full pipeline - calls retrieval + displays formatted results
#
# Parameters for print_top_results_and_scores:
#   - query (str): Search query string
#   - embeddings (torch.tensor): Chunk embeddings tensor (use text_chunk_embeddings)
#   - pages_and_chunks (list): Chunk records with metadata (use pages_and_chunks_over_min_token_len!)
#   - model (SentenceTransformer): Encoding model (use embedding_model)
#   - n_resources_to_return (int): Number of results to display (default 4)
#
# Output: Nicely formatted terminal output showing:
#   - Rank, similarity score
#   - Chunk text (word-wrapped for readability)
#   - Source page number
#
# IMPORTANT: Always pass pages_and_chunks_over_min_token_len (filtered) NOT pages_and_chunks (unfiltered)

import textwrap
import torch

def print_wrapped(text, wrap_length=80):
    """Wrap long text to specified line width for console display."""
    print(textwrap.fill(text, wrap_length))


def print_top_results_and_scores(
    query: str,
    embeddings: torch.tensor,
    pages_and_chunks: list,
    model,
    n_resources_to_return: int = 4
):
    """
    Retrieves most relevant chunks and displays them with scores and source pages.
    
    Args:
        query (str): Search query string
        embeddings (torch.tensor): Chunk embedding tensor (should be text_chunk_embeddings)
        pages_and_chunks (list): List of chunk records with metadata (should be pages_and_chunks_over_min_token_len!)
        model: Encoding model for query (use embedding_model)
        n_resources_to_return (int): Number of results to display (default 4)
    
    Returns:
        None (prints to console)
    
    Important:
        - Always pass pages_and_chunks_over_min_token_len (filtered by min_token_length)
        - NOT pages_and_chunks (unfiltered, some chunks may be too short)
    """

    # Call core retrieval function to get scores and indices
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        model=model,
        n_resources_to_return=n_resources_to_return,
        print_time=True
    )

    # Print header
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}\n")

    # Display each result ranked by similarity score
    for rank, (score, index) in enumerate(zip(scores, indices), 1):
        print(f"[Rank {rank}] Similarity Score: {score:.4f}")
        print("-" * 80)
        
        print("Chunk Text:")
        print_wrapped(pages_and_chunks[index]["sentence_chunk"])
        
        print(f"\nSource Page: {pages_and_chunks[index]['page_number']}")
        print(f"Tokens: {pages_and_chunks[index].get('chunk_token_count', 'N/A')}")
        print(f"{'='*80}\n")

In [ ]:
# Cell: Example Query - Complete Semantic Search Pipeline in Action
# Purpose: Demonstrate entire retrieval workflow from query → embedding → ranking → display
# Query: "symptoms of pellagra" (real clinical/nutrition question)
# 
# Steps shown:
#   1. Call retrieve_relevant_resources() to get top-5 ranked chunks
#   2. Display raw scores and indices from torch.topk
#   3. Call print_top_results_and_scores() for formatted output with text and source pages
#
# This cell can be reused by changing the query string at the top
# Note: Requires text_chunk_embeddings and pages_and_chunks_over_min_token_len to be populated

# Define a domain-specific query (from nutrition textbook)
query = "symptoms of pellagra"

print(f"\n{'='*80}")
print("SEMANTIC SEARCH PIPELINE DEMO")
print(f"{'='*80}\n")
print(f"Running semantic search for: '{query}'\n")

# Get raw scores + indices using core retrieval function
scores, indices = retrieve_relevant_resources(
    query=query,
    embeddings=text_chunk_embeddings,
    model=embedding_model,
    n_resources_to_return=5
)

print(f"Retrieved {len(indices)} relevant chunks")
if len(scores) > 0:
    print(f"Score range: {scores[-1]:.4f} to {scores[0]:.4f} (higher = more relevant)\n")

# Print formatted results with text and source pages
print("\n" + "="*80)
print("FORMATTED RESULTS WITH TEXT & SOURCE PAGES")
print("="*80)

print_top_results_and_scores(
    query=query,
    embeddings=text_chunk_embeddings,
    pages_and_chunks=pages_and_chunks_over_min_token_len,  # Use FILTERED list!
    model=embedding_model,
    n_resources_to_return=5
)

In [ ]:
# Cell: CPU-Based LLM Model Selection
# Purpose: Select appropriate small LLM for CPU inference
# Note: CPU inference requires smaller models (small parameters) and optimized loading
# Recommended models for CPU inference:
#   - distilgpt2 (82M params) - fastest, good for quick responses
#   - gpt2 (124M params) - balanced speed/quality  
#   - gpt2-medium (355M params) - better quality, slower
# Parameters used:
#   - torch_dtype: float32 (CPU-native, standard precision)
#   - device_map: 'cpu' (explicit CPU placement)

# Model selection for CPU inference
# Using distilgpt2: lightweight, efficient, good quality for CPU (82M parameters)
model_id = "distilgpt2"

print(f"✓ Model selected for CPU inference: {model_id}")
print(f"  Expected inference speed: ~100-200 tokens/second on CPU")
print(f"  Memory requirement: ~500MB-1GB")
print(f"  Loading model - this will download {model_id} (~350MB)...")
print(f"\nNote: Using CPU-only inference. For faster generation, install CUDA PyTorch and GPU.")

In [ ]:
# Cell: Hugging Face Authentication (Optional)
# Purpose: Authenticate with Hugging Face Hub for private models
# Note: distilgpt2 and other public models don't require authentication
# Only needed if using private/gated models from your account
#
# To enable authentication:
#   1. Get your HF token from https://huggingface.co/settings/tokens
#   2. Copy token below
#   3. Uncomment the login lines

from huggingface_hub import login

# Public models used (no auth needed)
# login(token="YOUR_HF_TOKEN_HERE")  # Uncomment if using private models

print("✓ Using public models (no authentication required)")

In [ ]:
# Cell: CPU-Based LLM Loading and Setup
# Purpose: Load tokenizer and LLM model for CPU-only inference
# Key settings for CPU optimization:
#   - torch_dtype=torch.float32: CPU-native precision (no conversion overhead)
#   - device_map='cpu': Explicit CPU placement (prevents CUDA attempts)
#   - low_cpu_mem_usage=True: Memory-efficient loading
# Model will run on pure CPU using standard PyTorch inference
#
# Performance: ~100-200 tokens/second depending on CPU cores and model size
# Warning: First generation may take 30-120 seconds as model runs on CPU

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

# Device setup (explicit CPU)
device = "cpu"
print(f"[INFO] Device set to: {device}")

# Load tokenizer (CPU-compatible)
print(f"[INFO] Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token for batch processing

print(f"✓ Tokenizer loaded successfully")

# Load LLM model for CPU inference
print(f"[INFO] Loading model {model_id} for CPU inference...")
print(f"      This may take 30-60 seconds on first load...")

llm_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    torch_dtype=torch.float32,  # CPU-native precision (float32 is standard)
    device_map=device,  # Explicitly place on CPU
    low_cpu_mem_usage=True,  # Memory-efficient loading
    trust_remote_code=True
)

# Move model to CPU (redundant but explicit)
llm_model = llm_model.to(device)

# Disable gradient computation for inference (saves memory and speeds up)
llm_model.eval()

# Print model info
total_params = sum(p.numel() for p in llm_model.parameters())
print(f"\n✓ Model loaded successfully on {device}")
print(f"  Model: {model_id}")
print(f"  Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"  Dtype: {llm_model.dtype}")
print(f"  Device: {next(llm_model.parameters()).device}")
print(f"\n⚠  CPU INFERENCE WARNING:")
print(f"  - First text generation will take 30-120 seconds")
print(f"  - Subsequent generations will be 100-200 tokens/second")
print(f"  - For faster inference, use GPU with CUDA PyTorch")

In [ ]:
# Cell: Model Statistics
# Purpose: Display loaded model information

print(f"\nModel Information:")
print(f"  Model ID: {model_id}")
print(f"  Total Parameters: {sum(p.numel() for p in llm_model.parameters()):,}")
print(f"  Data Type: {llm_model.dtype}")
print(f"  Device: {next(llm_model.parameters()).device}")
print(f"  Memory Mode: CPU-Only (no GPU acceleration)")
print(f"\nTokenizer Information:")
print(f"  Vocab Size: {len(tokenizer)}")
print(f"  Padding Token: {tokenizer.pad_token}")
print(f"  EOS Token: {tokenizer.eos_token}")

In [ ]:
def get_model_num_parameters(modelL: torch.nn.Module): 
    return sum(p.numel() for p in model.parameters())  

get_model_num_parameters(llm_model) 

In [ ]:
# Cell: Tokenization + Generation on CPU
# Purpose: Tokenize input text and generate output using CPU-based model
# Parameters:
#   - return_tensors='pt': Return PyTorch tensors
#   - max_new_tokens: 100 (number of tokens to generate - reduced for CPU speed)
#   - temperature: 0.7 (controls randomness)
# Note: CPU inference will take 30-60 seconds for first generation
#       Subsequent generations: ~100-200 tokens/second

import time

# Tokenize the input text and place on CPU
input_ids = tokenizer(prompt, return_tensors="pt").to("cpu")

print(f"[INFO] Prompt length: {input_ids['input_ids'].shape[1]} tokens")
print(f"[INFO] Generating response on CPU (may take 30-60 seconds)...\\n")

# Generate outputs using CPU
start_time = time.time()

with torch.no_grad():  # Disable gradient computation for faster inference
    outputs = llm_model.generate(
        **input_ids,
        max_new_tokens=100,  # Reduced for CPU speed (increase for longer responses)
        temperature=0.7,  # Controls randomness (0.7 = balanced)
        do_sample=True,  # Use sampling instead of greedy
        top_k=50,  # Keep top 50 tokens for diversity
        top_p=0.95,  # Nucleus sampling
        pad_token_id=tokenizer.eos_token_id
    )

elapsed = time.time() - start_time
tokens_generated = outputs.shape[1] - input_ids['input_ids'].shape[1]
print(f"[INFO] Generated {tokens_generated} tokens in {elapsed:.1f} seconds")
print(f"[INFO] Speed: {tokens_generated/elapsed:.1f} tokens/second\\n")

print(f"Model output (tokens):\\n{outputs[0][:50]}...  # First 50 token IDs")

In [ ]:
# Cell: Decode Model Output
# Purpose: Convert model output tokens back into readable text
# Parameters:
#   - skip_special_tokens=True: Remove <bos>, <eos>, padding tokens
#   - clean_up_tokenization_spaces=True: Clean up extra spaces

# Decode the output tokens to readable text
outputs_decoded = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

print(f"Model output (decoded):\\n{outputs_decoded}")

In [ ]:
# Cell: Clean Output Text
# Purpose: Remove prompt and special tokens from model output
# Output should contain only the generated text (not the input prompt)

# Clean output text by removing prompt and special characters
cleaned_output = (
    outputs_decoded
    .replace(prompt, "")  # Remove input prompt
    .replace("<bos>", "")  # Remove beginning-of-sequence token
    .replace("<eos>", "")  # Remove end-of-sequence token
    .strip()  # Remove leading/trailing whitespace
)

print("Generated text (cleaned):")
print("=" * 80)
print(cleaned_output)
print("=" * 80)

In [ ]:
# Query List Creation 
manual_questions = [
    "How often should infants be breastfed?",
    "What are symptoms of pellagra?",
    "How does saliva help with digestion?",
    "What is the RDI for protein per day?",
    "water soluble vitamins"
]

query_list = gpt4_questions + manual_questions

In [ ]:
# Retrieve Relevant Resources 
def retrieve_relevant_resources(
    query: str,
    embeddings: torch.tensor,
    model,
    n_resources_to_return: int = 5,
    print_time: bool = True
):
    """
    Embeds a query with model and returns top k scores and indices from embeddings.
    """

    # Embed the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Get dot product scores on embeddings
    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Time taken to get scores on {len(embeddings)} embeddings: {end_time - start_time:.5f} seconds.")

    scores, indices = torch.topk(input=dot_scores, k=n_resources_to_return)

    return scores, indices

In [ ]:
# Print Retrieved Results 
def print_top_results_and_scores(
    query: str,
    embeddings: torch.tensor,
    pages_and_chunks: list[dict],
    n_resources_to_return: int = 5
):
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=n_resources_to_return
    )

    print(f"Query: {query}\n")
    print("Results:")

    # Loop through zipped together scores and indices
    for score, index in zip(scores, indices):
        print(f"Score: {score:.4f}")

        # Print relevant sentence chunk
        print_wrapped(pages_and_chunks[index]["sentence_chunk"])

        # Print page number for reference
        print(f"Page number: {pages_and_chunks[index]['page_number']}")
        print("\n")

In [ ]:
# Test Retrieval 
import random

query = random.choice(query_list)
print(f"Query: {query}")

scores, indices = retrieve_relevant_resources(
    query=query,
    embeddings=embeddings
) 

In [ ]:
# Full RAG Flow (Context + Prompt Formatting) 
# Create context items
context_items = [pages_and_chunks[i] for i in indices]

# Format prompt with context
prompt = prompt_formatter(
    query=query,
    context_items=context_items
)

print(prompt) 

In [ ]:
# Prompt Formatter 
def prompt_formatter(query: str, context_items: list[dict]) -> str:
    """
    Augments query with text-based context from context_items.
    """

    # Join context items into one paragraph
    context = "- " + "\n- ".join(
        [item["sentence_chunk"] for item in context_items]
    )

    base_prompt = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Make sure your answers are as explanatory as possible.
Use the following examples as reference for the ideal answer style.

Example 1:
Query: What are the fat-soluble vitamins?
Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, Vitamin E, and Vitamin K.

Example 2:
Query: What are the causes of type 2 diabetes?
Answer: Type 2 diabetes is often associated with overnutrition.

Example 3:
Query: What is the importance of hydration for physical performance?
Answer: Hydration is crucial for physical performance.

Now use the following context items to answer the user query:
{context}

User query: {query}
Answer:
"""

    return base_prompt.format(context=context, query=query) 

In [ ]:
# Final Generation Step 
# # Tokenize prompt
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate output tokens
outputs = llm_model.generate(
    **input_ids,
    temperature=0.7,          # lower = deterministic, higher = creative
    do_sample=True,           # enable sampling
    max_new_tokens=256        # number of tokens to generate
)

# Decode output
output_text = tokenizer.decode(outputs[0])

print(f"Query: {query}")
print(f"RAG answer:\n{output_text.replace(prompt, '')}")

In [ ]:
# Cell: CPU-Based Full RAG Wrapper Function
# Purpose: Complete RAG pipeline - retrieve context and generate answer on CPU
# Pipeline:
#   1. Retrieve top-k relevant chunks using semantic search
#   2. Format prompt with context  
#   3. Tokenize on CPU
#   4. Generate answer using CPU-based LLM
#   5. Clean and return output
# Parameters:
#   - temperature: 0.7 (balanced randomness)
#   - max_new_tokens: 256 (length of generated answer - reduced for CPU)
#   - format_answer_text: Clean up special tokens
#   - return_answer_only: Return just answer or answer + context
# Warning: CPU inference is slow. For 256 tokens, expect 60-120 seconds
#
# Example usage:
#   answer = ask("What are symptoms of pellagra?", max_new_tokens=256)

import time

def ask(
    query,
    temperature=0.7,
    max_new_tokens=256,
    format_answer_text=True,
    return_answer_only=True
):
    """
    Takes a query, retrieves context from documents, and generates an answer using CPU-based LLM.
    
    Args:
        query (str): User's question  
        temperature (float): Controls randomness (0.0-1.0, higher = more creative)
        max_new_tokens (int): Maximum tokens to generate (lower = faster on CPU)
        format_answer_text (bool): Clean up special tokens and formatting
        return_answer_only (bool): Return only answer or answer + context
    
    Returns:
        str or tuple: If return_answer_only=True, returns just the answer text.
                      Otherwise returns (answer_text, context_items)
    """
    
    print(f"\\n[RAG PIPELINE] Processing query: {query[:50]}...")
    
    # Step 1: Retrieve top-k relevant chunks using semantic search
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=text_chunk_embeddings,
        model=embedding_model,
        n_resources_to_return=5,
        print_time=True
    )
    
    # Step 2: Build context from retrieved chunks
    context_items = [pages_and_chunks_over_min_token_len[i] for i in indices]
    
    # Attach retrieval scores for debugging
    for i, item in enumerate(context_items):
        item["retrieval_score"] = scores[i].item()
    
    print(f"[INFO] Retrieved {len(context_items)} context chunks")
    
    # Step 3: Format prompt with context
    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )
    
    # Step 4: Tokenize on CPU
    input_ids = tokenizer(prompt, return_tensors="pt").to("cpu")
    print(f"[INFO] Total tokens: {input_ids['input_ids'].shape[1]} (context + query)")
    
    # Step 5: Generate answer using CPU-based LLM
    print(f"[INFO] Generating answer on CPU (this may take 30-120 seconds)...")
    start_time = time.time()
    
    with torch.no_grad():  # Disable gradients for faster inference
        outputs = llm_model.generate(
            **input_ids,
            temperature=temperature,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    elapsed = time.time() - start_time
    tokens_per_sec = max_new_tokens / elapsed if elapsed > 0 else 0
    print(f"[INFO] Generation completed in {elapsed:.1f}s ({tokens_per_sec:.1f} tokens/sec)")
    
    # Step 6: Decode output
    output_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    
    # Step 7: Clean output
    if format_answer_text:
        output_text = (
            output_text
            .replace(prompt, "")  # Remove prompt from output
            .strip()
        )
    
    # Step 8: Return result
    if return_answer_only:
        return output_text
    
    return output_text, context_items

In [ ]:
# Cell: Test CPU-Based RAG Pipeline
# Purpose: Test complete RAG pipeline with CPU inference
# Warning: First execution will take 1-2 minutes as model loads and runs on CPU
#
# Steps demonstrated:
#   1. Select a random query
#   2. Retrieve relevant context chunks
#   3. Generate answer using CPU-based LLM
#   4. Display timing and results

import random
import time

# Select a random query from evaluation set
query = random.choice(eval_questions)
print(f"Random Query: {query}\\n")

# Run complete RAG pipeline on CPU
start_time = time.time()
answer, context_items = ask(
    query=query,
    temperature=0.7,
    max_new_tokens=256,
    return_answer_only=False
)
total_time = time.time() - start_time

print(f"\\n{'='*80}")
print(f"RESULTS (Total time: {total_time:.1f} seconds)")
print(f"{'='*80}")

print(f"\\nGenerated Answer:")
print("-" * 80)
print_wrapped(answer)

print(f"\\n\\nContext Items Used (Retrieved via semantic search):")
print("-" * 80)
for i, item in enumerate(context_items, 1):
    score = item.get('retrieval_score', 'N/A')
    page = item.get('page_number', '?')
    tokens = item.get('chunk_token_count', '?')
    print(f"[{i}] Score: {score:.4f} | Page: {page:3d} | Tokens: {tokens:4d}")
    print(f"     Text: {item['sentence_chunk'][:70]}...")
    print()

In [ ]:
# Cell: CPU-Compatible RAG Evaluation Setup
# Purpose: Set up evaluation framework for RAG system (CPU-compatible)
# Note: RAGAS metrics are all CPU-compatible but compute-intensive
# For faster evaluation on CPU, reduce number of test questions
#
# If RAGAS not installed, install with: pip install ragas

import pandas as pd
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Try to import RAGAS (optional, evaluation can work without it)
try:
    from ragas import evaluate
    from ragas.metrics import (
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness
    )
    ragas_available = True
    print("✓ RAGAS library available - evaluation enabled")
except ImportError:
    ragas_available = False
    print("⚠ RAGAS not installed. Evaluation will be skipped.")
    print("  Install with: pip install ragas")

# Optional metrics (may require additional dependencies)
context_entity_recall = None
noise_robustness = None

if ragas_available:
    try:
        from ragas.metrics import context_entity_recall
        print("✓ Optional metric available: context_entity_recall")
    except ImportError:
        pass
    
    try:
        from ragas.metrics import noise_robustness
        print("✓ Optional metric available: noise_robustness")
    except ImportError:
        pass

print("\\n✓ Evaluation setup complete (CPU-compatible)")

In [ ]:
# Cell: CPU-Friendly Evaluation Function
# Purpose: Evaluate RAG performance with fallback options
# Note: For faster evaluation on CPU, reduce 'sample_size'

def evaluate_rag_performance(
    retriever,
    llm,
    embeddings,
    test_questions,
    reference_answers=None,
    sample_size=None,
    use_ragas=True
):
    """
    Evaluate RAG system performance.
    
    Args:
        retriever: Document retriever
        llm: Language model for answer generation
        embeddings: Embeddings model
        test_questions: List of test questions
        reference_answers: List of reference answers (optional)
        sample_size: Limit evaluation to N questions (for faster CPU runs)
        use_ragas: Use RAGAS metrics if available (CPU-intensive)
    
    Returns:
        dict: Evaluation results
    """
    
    if sample_size and sample_size < len(test_questions):
        test_questions = test_questions[:sample_size]
        if reference_answers:
            reference_answers = reference_answers[:sample_size]
        print(f"Evaluating on {sample_size} samples (CPU optimization)")
    
    results = {
        "total_questions": len(test_questions),
        "metrics": {},
        "sample_results": []
    }
    
    # If RAGAS is available, use it; otherwise fall back to basic metrics
    if use_ragas and ragas_available:
        try:
            data_samples = []
            
            for i, question in enumerate(test_questions):
                # Retrieve context
                docs = retriever.get_relevant_documents(question)
                context = "\n\n".join([doc.page_content for doc in docs])
                
                # Generate answer
                response = llm.invoke(f"Question: {question}\nContext: {context}\n\nAnswer:")
                answer = response.content if hasattr(response, 'content') else str(response)
                
                data_samples.append({
                    "question": question,
                    "contexts": [context],
                    "answer": answer,
                    "ground_truth": reference_answers[i] if reference_answers else answer
                })
            
            # Create RAGAS dataset
            rag_dataset = Dataset.from_dict({
                "question": [s["question"] for s in data_samples],
                "contexts": [s["contexts"] for s in data_samples],
                "answer": [s["answer"] for s in data_samples],
                "ground_truth": [s["ground_truth"] for s in data_samples]
            })
            
            # Evaluate with RAGAS (CPU-compatible, but compute-intensive)
            print("Running RAGAS evaluation (this may take a few minutes on CPU)...")
            ragas_results = evaluate(
                rag_dataset,
                metrics=[
                    context_precision,
                    context_recall,
                    answer_relevancy,
                    faithfulness
                ]
            )
            
            results["metrics"] = {
                "context_precision": float(ragas_results["context_precision"]),
                "context_recall": float(ragas_results["context_recall"]),
                "answer_relevancy": float(ragas_results["answer_relevancy"]),
                "faithfulness": float(ragas_results["faithfulness"])
            }
            results["sample_results"] = data_samples
            
        except Exception as e:
            print(f"RAGAS evaluation failed: {e}")
            print("Falling back to basic metrics...")
            results = evaluate_rag_basic(retriever, llm, test_questions, reference_answers)
    else:
        print("Using basic evaluation metrics (RAGAS not available)...")
        results = evaluate_rag_basic(retriever, llm, test_questions, reference_answers)
    
    return results


def evaluate_rag_basic(retriever, llm, test_questions, reference_answers=None):
    """Basic RAG evaluation without RAGAS."""
    results = {"total_questions": len(test_questions), "metrics": {}, "sample_results": []}
    
    total_precision = 0
    total_relevancy = 0
    
    for i, question in enumerate(test_questions):
        docs = retriever.get_relevant_documents(question)
        context = "\n\n".join([doc.page_content for doc in docs][:3])  # Top 3 docs
        
        response = llm.invoke(f"Question: {question}\nContext: {context}\n\nAnswer:")
        answer = response.content if hasattr(response, 'content') else str(response)
        
        # Simple heuristics
        context_precision = min(1.0, len(docs) / 5)  # Assuming 5+ docs is good
        answer_relevancy = 0.7 if len(answer) > 50 else 0.4  # Longer answers tend to be more relevant
        
        total_precision += context_precision
        total_relevancy += answer_relevancy
        
        results["sample_results"].append({
            "question": question,
            "answer": answer,
            "num_docs_retrieved": len(docs)
        })
    
    results["metrics"] = {
        "avg_context_precision": total_precision / len(test_questions),
        "avg_answer_relevancy": total_relevancy / len(test_questions),
        "note": "Basic metrics (RAGAS unavailable)"
    }
    
    return results

print("✓ Evaluation functions defined")

In [ ]:
# Generate Evaluation Data
evaluation_data = []

print("Generating RAG answers for evaluation...")

for question, ground_truth in zip(eval_questions, ground_truth_answers):
    print(f"Processing: {question[:50]}...")

    rag_answer, contexts = ask(
        query=question,
        return_answer_only=False
    )

    evaluation_data.append({
        "question": question,
        "answer": rag_answer,
        "contexts": [c["sentence_chunk"] for c in contexts],
        "ground_truth": ground_truth
    })

In [ ]:
# Run RAGAS Evaluation 
# Convert to dataset
eval_dataset = Dataset.from_pandas(pd.DataFrame(evaluation_data))

# Metrics
metrics = [
    context_precision,
    context_recall,
    answer_relevancy,
    faithfulness
]

# Add optional metrics if available
if context_entity_recall is not None:
    metrics.append(context_entity_recall)

if noise_robustness is not None:
    metrics.append(noise_robustness)

# Run evaluation
print("Running RAGAS evaluation...")
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics
)

# Convert to DataFrame
results_df = results.to_pandas()
results_df

In [ ]:
# Extract Numeric Metrics Columns 
metric_cols = []

for col in results_df.columns:
    if col not in ['user_input', 'retrieved_contexts', 'response', 'reference']:
        try:
            numeric_values = pd.to_numeric(results_df[col], errors='coerce')
            if not numeric_values.isna().all():
                metric_cols.append(col)
        except:
            pass

In [ ]:
# Individual Question Performance Table 
print("\n📊 INDIVIDUAL QUESTION PERFORMANCE")
print("-" * 80)

# Clean DataFrame for display
clean_results = pd.DataFrame()

clean_results["question"] = results_df["user_input"]

for col in metric_cols:
    clean_results[col] = results_df[col]

clean_results

In [ ]:
#Compute Overall Average Scores 
print("\n📈 OVERALL AVERAGE SCORES")
print("-" * 50)

avg_scores = {}

for col in metric_cols:
    avg_score = results_df[col].mean()
    avg_scores[col] = avg_score

    # Performance classification
    if avg_score >= 0.8:
        indicator = "Excellent"
    elif avg_score >= 0.6:
        indicator = "Good"
    elif avg_score >= 0.4:
        indicator = "Fair"
    else:
        indicator = "Poor"

    print(f"{col.replace('_', ' ').title():<25}: {avg_score:.3f}  ({indicator})")

In [ ]:
#Performance Summary Counts 
print("\n📊 PERFORMANCE SUMMARY")
print("-" * 50)

excellent = sum(1 for score in avg_scores.values() if score >= 0.8)
good = sum(1 for score in avg_scores.values() if 0.6 <= score < 0.8)
fair = sum(1 for score in avg_scores.values() if 0.4 <= score < 0.6)
poor = sum(1 for score in avg_scores.values() if score < 0.4)

print(f"Excellent metrics: {excellent}")
print(f"Good metrics:      {good}")
print(f"Fair metrics:      {fair}")
print(f"Poor metrics:      {poor}")

In [ ]:
# Key Insights
print("\n🔍 KEY INSIGHTS")
print("-" * 50)

best_metric = max(avg_scores, key=avg_scores.get)
worst_metric = min(avg_scores, key=avg_scores.get)

print(f"Best performing:  {best_metric.replace('_',' ').title()} ({avg_scores[best_metric]:.3f})")
print(f"Needs improvement: {worst_metric.replace('_',' ').title()} ({avg_scores[worst_metric]:.3f})")

In [ ]:
#Automated Diagnosis + Recommendations 
# Diagnose issues based on metrics

if avg_scores.get('context_precision', 0) > 0.7 and avg_scores.get('answer_relevancy', 0) < 0.3:
    print("\n⚠️ Issue detected: Good retrieval but poor answer generation")
    print("→ Fix your prompt template to better use retrieved context")

if avg_scores.get('context_recall', 0) < 0.5:
    print("\n⚠️ Issue detected: Poor context retrieval")
    print("→ Improve embedding model or chunking strategy")

In [ ]:
#Save Results to CSV 
print("\n💾 Saving results...")

detailed_results = pd.DataFrame(evaluation_data)

for col in metric_cols:
    detailed_results[col] = results_df[col].values

detailed_results.to_csv("rag_evaluation_results.csv", index=False)

print("Saved to rag_evaluation_results.csv")
print("=" * 80)
